# 01 · Memahami dan Membersihkan Data Book-Crossing

**Proyek ADL — Sistem Rekomendasi Buku (Hybrid Switching)**

Notebook ini adalah **langkah 1** dari 4:

| Notebook | Isi |
|---|---|
| **01_Data_Cleaning** | membaca data mentah, menemukan masalah, membersihkan |
| 02_EDA | eksplorasi data dan insight |
| 03_Pemodelan | filter & split, lalu membangun setiap model satu per satu |
| 04_Evaluasi_dan_Hybrid | evaluasi, tuning hybrid switching, menyimpan model untuk dashboard |

**Dataset.** Book-Crossing (Ziegler dkk., 2005) dikumpulkan dari komunitas tukar-buku *bookcrossing.com* pada Agustus–September 2004. Ada tiga tabel:

- `BX-Users.csv` — User-ID, Location ("kota, provinsi, negara"), Age
- `BX-Books.csv` — ISBN, judul, penulis, tahun terbit, penerbit, URL gambar sampul
- `BX-Book-Ratings.csv` — User-ID, ISBN, rating **0–10** (0 = *implicit*, 1–10 = *explicit*)

> **Cara menjalankan:** letakkan ketiga CSV di `data/raw/`, lalu jalankan notebook ini dari folder `notebooks/`.

In [1]:
import sys
sys.path.append("..")          # agar folder src/ bisa di-import

import numpy as np
import pandas as pd
from src import config as C
from src import data_prep as D

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 160)

## 1. Membaca data mentah — dan jebakan pertamanya

File memakai pemisah `;` dan encoding `latin-1`. Cara "biasa" di bawah tampak berhasil, tetapi sebenarnya ada baris yang **rusak atau hilang**.

In [2]:
naive = pd.read_csv(C.RAW_FILES["books"], sep=";", encoding="latin-1", dtype=str, on_bad_lines="skip")
print("Jumlah baris (cara biasa):", len(naive))

# Tahun terbit seharusnya angka. Baris yang tahunnya bukan angka = kolomnya bergeser
geser = naive[~naive["Year-Of-Publication"].str.fullmatch(r"\d+", na=False)]
geser.iloc[:, :5]

Jumlah baris (cara biasa): 271360


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
209538,078946697X,"DK Readers: Creating the X-Men, How It All Began (Level ...",2000,DK Publishing Inc,http://images.amazon.com/images/P/078946697X.01.THUMBZZZ...
220731,2070426769,"Peuple du ciel, suivi de 'Les Bergers\"";Jean-Marie Gusta...",2003,Gallimard,http://images.amazon.com/images/P/2070426769.01.THUMBZZZ...
221678,0789466953,"DK Readers: Creating the X-Men, How Comic Books Come to ...",2000,DK Publishing Inc,http://images.amazon.com/images/P/0789466953.01.THUMBZZZ...


Pada 3 baris di atas, judul memuat tanda kutip yang di-*escape* (`\"`). Akibatnya nama penulis "terdorong" ke kolom judul, tahun ke kolom penulis, dan seterusnya. Selain itu ada 19 baris yang **dilewati diam-diam** oleh `on_bad_lines="skip"`.

**Solusinya:** beri tahu pandas bahwa karakter escape-nya adalah backslash (`escapechar="\\"`). Fungsi `D.read_raw()` sudah melakukannya.

In [3]:
books_raw   = D.read_raw("books")
users_raw   = D.read_raw("users")
ratings_raw = D.read_raw("ratings")

for name, df in [("Books", books_raw), ("Users", users_raw), ("Ratings", ratings_raw)]:
    print(f"{name:8s}: {df.shape[0]:>9,} baris x {df.shape[1]} kolom")

books_raw[books_raw["ISBN"].isin(geser["ISBN"])].iloc[:, :5]

Books   :   271,379 baris x 8 kolom
Users   :   278,858 baris x 3 kolom
Ratings : 1,149,780 baris x 3 kolom


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
209550,078946697X,"DK Readers: Creating the X-Men, How It All Began (Level ...",Michael Teitelbaum,2000,DK Publishing Inc
220744,2070426769,"Peuple du ciel, suivi de 'Les Bergers""",Jean-Marie Gustave Le ClÃ?Â©zio,2003,Gallimard
221691,0789466953,"DK Readers: Creating the X-Men, How Comic Books Come to ...",James Buckley,2000,DK Publishing Inc


Sekarang semua **271.379** buku terbaca dan ketiga baris tadi kolomnya sudah benar.

## 2. Tabel Users

In [4]:
users_raw.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17
4,5,"farnborough, hants, united kingdom",NaN


In [5]:
age = pd.to_numeric(users_raw["Age"], errors="coerce")
print(f"Age kosong       : {age.isna().mean():.1%}")
print(f"Age < {C.AGE_MIN}          : {(age < C.AGE_MIN).sum():,} user")
print(f"Age > {C.AGE_MAX}        : {(age > C.AGE_MAX).sum():,} user")
age.describe()

Age kosong       : 39.7%
Age < 5          : 882 user
Age > 100        : 366 user


count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64

Ada usia 0 dan 244 tahun — jelas tidak valid. Usia di luar **5–100** diubah menjadi *missing* (bukan dibuang usernya, karena rating mereka tetap berguna).

Kolom `Location` berformat "kota, provinsi, negara"; bagian terakhir diambil sebagai **negara**.

In [6]:
users = D.clean_users(users_raw)
print(f"Age kosong setelah cleaning: {users['age'].isna().mean():.1%}")
users.head()

Age kosong setelah cleaning: 40.2%


,user_id,location,country,age
0,1,"nyc, new york, usa",usa,<NA>
1,2,"stockton, california, usa",usa,18
2,3,"moscow, yukon territory, russia",russia,<NA>
3,4,"porto, v.n.gaia, portugal",portugal,17
4,5,"farnborough, hants, united kingdom",united kingdom,<NA>


In [7]:
users["country"].value_counts().head(10)

country
usa               139713
canada             21658
united kingdom     18557
germany            17073
spain              13258
australia          11785
italy              11271
france              3487
portugal            3390
new zealand         3120
Name: count, dtype: int64

## 3. Tabel Books

Masalah yang ditangani `D.clean_books()`:

1. **ISBN** diseragamkan: huruf besar, hanya karakter `0-9` dan `X`.
2. **Entitas HTML** di judul (`&amp;` → `&`), tanda kutip sisa, dan spasi berlebih dibersihkan.
3. **Tahun terbit** 0 atau di luar 1800–2006 dijadikan *missing* (data dikumpulkan tahun 2004).

In [8]:
yr = pd.to_numeric(books_raw["Year-Of-Publication"], errors="coerce")
print("Tahun = 0           :", (yr == 0).sum())
print("Tahun > 2006        :", (yr > 2006).sum(), "->", sorted(int(x) for x in yr[yr > 2006].unique()))
print("Judul berisi &amp;  :", books_raw["Book-Title"].str.contains("&amp;").sum())

books_isbn = D.clean_books(books_raw)
print("\nSetelah cleaning:", f"{len(books_isbn):,} ISBN unik")
books_isbn.head(3)

Tahun = 0           : 4619
Tahun > 2006        : 23 -> [2008, 2010, 2011, 2012, 2020, 2021, 2024, 2026, 2030, 2037, 2038, 2050]
Judul berisi &amp;  : 4884

Setelah cleaning: 271,016 ISBN unik


,isbn,title,author,year,publisher,img_m,img_l
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.01.MZZZZZZZ...,http://images.amazon.com/images/P/0195153448.01.LZZZZZZZ...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.01.MZZZZZZZ...,http://images.amazon.com/images/P/0002005018.01.LZZZZZZZ...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.01.MZZZZZZZ...,http://images.amazon.com/images/P/0060973129.01.LZZZZZZZ...


## 4. Tabel Ratings

In [9]:
r = ratings_raw["Book-Rating"].astype(int)
print(f"Total rating      : {len(r):,}")
print(f"Rating 0 (implicit): {(r == 0).sum():,}  ({(r == 0).mean():.1%})")
print(f"Rating 1-10        : {(r > 0).sum():,}")
r.value_counts().sort_index().to_frame("jumlah").T

Total rating      : 1,149,780
Rating 0 (implicit): 716,109  (62.3%)
Rating 1-10        : 433,671


Book-Rating,0,1,2,3,4,5,6,7,8,9,10
jumlah,716109,1770,2759,5996,8904,50974,36924,76457,103736,67541,78610


**Penting:** rating **0 bukan "sangat tidak suka"**. Nilai 0 berarti user berinteraksi dengan buku (membaca/menyimpan) tetapi **tidak memberi nilai**. Jadi:

- untuk **prediksi rating** → hanya rating 1–10 yang dipakai;
- untuk **association rules** → rating 0 ikut dipakai sebagai tanda "pernah membaca".

Masalah lain: sebagian rating merujuk ISBN yang **tidak ada** di tabel Books. Rating seperti itu tidak bisa dihubungkan ke judul/penulis, jadi dibuang.

In [10]:
ratings_isbn = D.clean_ratings(ratings_raw, books_isbn)
print(f"Rating dengan ISBN dikenal : {len(ratings_isbn):,}")
print(f"Rating dibuang (ISBN asing): {len(ratings_raw) - len(ratings_isbn):,}")

Rating dengan ISBN dikenal : 1,031,624
Rating dibuang (ISBN asing): 118,156


## 5. Satu judul, banyak ISBN — menggabungkan edisi

ISBN adalah kode per **edisi** (hardcover, paperback, cetakan ulang, large print, …). Bagi pembaca, edisi berbeda tetap **buku yang sama**. Jika tidak digabung:

- data rating terpecah sehingga matriks makin jarang;
- model bisa "merekomendasikan" edisi lain dari buku yang sudah dibaca, dan evaluasi jadi **terlalu optimis**.

In [11]:
contoh = books_isbn[books_isbn["title"].str.contains("Lovely Bones", case=False)
                    & books_isbn["author"].str.contains("Sebold", case=False)]
contoh[["isbn", "title", "author", "year", "publisher"]]

,isbn,title,author,year,publisher
408,0316666343,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown"
474,0316168815,The Lovely Bones,Alice Sebold,2004,Back Bay Books
88454,1402521154,The Lovely Bones,Alice Sebold,2002,Recorded Books
162533,159413023X,The Lovely Bones,Alice Sebold,2004,Large Print Press
192150,1402532903,The Lovely Bones,Alice Sebold,2002,Recorded Books
267441,1586638661,The Lovely Bones (Reader's Companion),Alice Sebold,2003,Barnes & Noble Books-Imports


Kunci "karya" dibuat dari **judul + penulis yang dinormalisasi** (`D.work_key`): huruf kecil, isi tanda kurung dibuang, subjudul generik seperti ": A Novel" dibuang, kata sandang awal (*the/a/an*) dibuang, dan tanda baca dihapus.

In [12]:
D.work_key(contoh["title"], contoh["author"]).to_frame("kunci karya")

,kunci karya
408,lovely bones|alicesebold
474,lovely bones|alicesebold
88454,lovely bones|alicesebold
162533,lovely bones|alicesebold
192150,lovely bones|alicesebold
267441,lovely bones|alicesebold


In [13]:
books, ratings = D.merge_editions(books_isbn, ratings_isbn)
print(f"ISBN unik         : {len(books_isbn):,}")
print(f"Judul (karya) unik: {len(books):,}   ->  {len(books_isbn) - len(books):,} ISBN digabung")
print(f"Rating            : {len(ratings_isbn):,} -> {len(ratings):,} "
      f"(user yang menilai >1 edisi judul sama dirata-rata)")
books.sort_values("n_editions", ascending=False).head(8)[["book_id", "title", "author", "n_editions"]]

ISBN unik         : 271,016
Judul (karya) unik: 232,059   ->  38,957 ISBN digabung
Rating            : 1,031,624 -> 1,024,060 (user yang menilai >1 edisi judul sama dirata-rata)


,book_id,title,author,n_editions
3198,0553210793,The Adventures of Huckleberry Finn (Bantam Classics),MARK TWAIN,52
123546,0671517643,LITTLE WOMEN,Louisa May Alcott,47
229657,0553212583,Wuthering Heights,EMILY BRONTE,46
164935,0553213105,Pride and Prejudice,Jane Austen,42
107380,0553211404,Jane Eyre (Bantam Classics),Charlotte Bronte,42
58649,0451523377,Dracula (Signet Classics (Paperback)),Bram Stoker,41
180893,006440188X,The Secret Garden,Frances Hodgson Burnett,41
86033,0140430032,Great Expectations (English Library),Charles Dickens,40


`book_id` = ISBN dari edisi yang **paling banyak dinilai** pada judul tersebut; dipakai sebagai ID buku untuk seluruh tahap berikutnya.

## 6. Ringkasan cleaning

Semua langkah di atas dibungkus dalam satu fungsi `D.load_clean()`, yang juga dipakai oleh pipeline dan dashboard.

In [14]:
books, users, ratings, summary = D.load_clean()
pd.Series(summary).map(lambda v: f"{v:,.0f}" if v > 100 else f"{v:.1f}").to_frame("nilai")

,nilai
books_raw,"271,379"
users_raw,"278,858"
ratings_raw,"1,149,780"
isbn_clean,"271,016"
books_clean,"232,059"
users_clean,"278,858"
ratings_isbn_known,"1,031,624"
ratings_clean,"1,024,060"
ratings_dropped_unknown_isbn,"118,156"
explicit,"382,359"


In [15]:
C.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
books.to_csv(C.PROCESSED_DIR / "books_clean.csv.gz", index=False)
users.to_csv(C.PROCESSED_DIR / "users_clean.csv.gz", index=False)
ratings.to_csv(C.PROCESSED_DIR / "ratings_clean.csv.gz", index=False)
print("Tersimpan di", C.PROCESSED_DIR)

Tersimpan di data/processed


### Poin untuk laporan

| Masalah | Penanganan |
|---|---|
| Tanda kutip ter-*escape* membuat 3 baris bergeser & 19 baris hilang | `escapechar="\\"` saat membaca |
| ±40% usia kosong; ada usia 0 dan >100 | usia di luar 5–100 → *missing* |
| Tahun terbit 0 atau di masa depan | tahun di luar 1800–2006 → *missing* |
| ±10% rating merujuk ISBN yang tidak ada di katalog | dibuang |
| Rating 0 = implicit, bukan rating buruk | dipisah: hanya 1–10 untuk prediksi rating |
| Satu judul punya banyak ISBN (sampai 52 edisi) | digabung menjadi satu `book_id` |

Lanjut ke **02_EDA**.